In [41]:
import ee
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Configuration from your .ipynb

BBOX = [8.8, 55.8, 9.4, 56.1] 

PATCH_SIZE = 13
START_DATE = "2015-07-01"
END_DATE = "2020-01-01"
PATCH_CENTER = PATCH_SIZE // 2
AOI = ee.Geometry.Rectangle(BBOX)

In [42]:
import os

def load_ismn(file_path):
    with open(file_path, 'r') as f:
        header = f.readline().strip().split()

    try:
        network = header[1]
        station = header[2]
        lat, lon = float(header[3]), float(header[4])
    except (IndexError, ValueError):
        return None

    # Use the folder name (e.g. 1.01) as physical location id so
    # multiple sensors under the same location can be aggregated.
    location_id = os.path.basename(os.path.dirname(file_path))

    df = pd.read_csv(
        file_path, 
        sep=r'\s+', 
        skiprows=1, 
        header=None, 
        engine='python'
    )

    if df.shape[1] < 4:
        return None

    df = df.iloc[:, :4]
    df.columns = ['date', 'time', 'sm', 'flag']

    # Filter for Good quality
    df = df[df['flag'] == 'G']
    if df.empty:
        return None

    # Cleaning
    df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'])
    df['sm'] = pd.to_numeric(df['sm'], errors='coerce')
    df = df.dropna(subset=['sm'])

    # Unit conversion (Percentage to m3/m3)
    if df['sm'].mean() > 1.0:
        df['sm'] = df['sm'] / 100.0

    # point_id now represents physical site location, not sensor id.
    df['point_id'] = f"{network}_{location_id}"
    df['sensor_id'] = station
    df['latitude'] = lat
    df['longitude'] = lon

    return df[['point_id', 'sensor_id', 'timestamp', 'sm', 'latitude', 'longitude']]

# --- EXECUTION ---
stm_files = glob.glob('Data/Denmark/HOBE/*/*.stm', recursive=True)

dfs = [load_ismn(f) for f in stm_files]
dfs = [d for d in dfs if d is not None]
df_combined = pd.concat(dfs, ignore_index=True)

# 1. MULTI-SENSOR COLLAPSE (Handle A, B, C duplicates by location)
ground_truth = df_combined.groupby(
    ['point_id', 'timestamp', 'latitude', 'longitude'], as_index=False
)['sm'].mean()

# 2. DAILY AGGREGATION
# Now we move from hourly/sub-hourly to daily
ground_truth['timestamp'] = (
    ground_truth['timestamp']
    .dt.tz_localize(
        'Europe/Copenhagen',
        nonexistent='shift_forward',
        ambiguous='NaT'
    )
    .dropna()
    .dt.tz_convert('UTC')
    .dt.floor('D')
)

ground_truth = ground_truth.groupby(
    ['point_id', 'timestamp', 'latitude', 'longitude'], as_index=False
)['sm'].mean()
print(f"Final ground truth shape: {ground_truth.shape}")
print(f"Unique locations: {ground_truth['point_id'].nunique()}")

Final ground truth shape: (61284, 5)
Unique locations: 30


In [43]:
# ================= CONFIG =================
TARGET_SCALE = 1000
TARGET_CRS = 'EPSG:32632'
PATCH_RADIUS = (PATCH_SIZE - 1) // 2


# ================= DATA LOADERS =================

def get_ndvi(aoi, start, end):
    col = ee.ImageCollection("MODIS/061/MOD13A1") \
        .filterBounds(aoi) \
        .filterDate(start, end)

    def proc(img):
        ndvi = img.select("NDVI").multiply(0.0001).rename("NDVI")
        return ndvi.copyProperties(img, ["system:time_start"])

    return col.map(proc)


def get_s1(aoi, start, end, speckle_radius_px=1):
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(aoi) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    def proc(img):
        bands = img.bandNames()

        vv = ee.Image(
            ee.Algorithms.If(bands.contains('VV'), img.select('VV'), ee.Image.constant(-20))
        ).toFloat().unmask(ee.Image.constant(-20))

        vh = ee.Image(
            ee.Algorithms.If(bands.contains('VH'), img.select('VH'), vv)
        ).toFloat().unmask(ee.Image.constant(-20))

        # Speckle suppression before deriving ratio.
        vv_f = vv.focal_median(radius=speckle_radius_px, units='pixels')
        vh_f = vh.focal_median(radius=speckle_radius_px, units='pixels')

        angle = img.select('angle').toFloat().unmask(ee.Image.constant(35))
        ratio = vv_f.subtract(vh_f).rename('VV_VH_Ratio')

        return ee.Image.cat([
            vv_f.rename('VV'),
            vh_f.rename('VH'),
            ratio,
            angle.rename('angle')
        ]).copyProperties(img, ['system:time_start'])

    return s1.map(proc)


def get_lst(aoi, start, end):
    return ee.ImageCollection('MODIS/061/MOD11A1') \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select('LST_Day_1km')
             .multiply(0.02).subtract(273.15)
             .rename('LST')
             .copyProperties(img, ['system:time_start']))


def get_smap(aoi, start, end):
    # SMAP anchor: direct microwave soil-moisture observation proxy.
    return ee.ImageCollection('NASA/SMAP/SPL4SMGP/008') \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select('sm_surface')
             .rename('SMAP')
             .copyProperties(img, ['system:time_start']))


def get_albedo(aoi, start, end):
    return ee.ImageCollection('MODIS/061/MCD43A3') \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select('Albedo_BSA_shortwave')
             .multiply(0.001)
             .rename('Albedo')
             .copyProperties(img, ['system:time_start']))


def add_vpd(img):
    img = ee.Image(img)
    T = img.select("T_air")
    Td = img.select("Td_air")

    es = T.expression(
        "0.6108 * exp((17.27 * T) / (T + 237.3))",
        {"T": T}
    )

    ea = Td.expression(
        "0.6108 * exp((17.27 * Td) / (Td + 237.3))",
        {"Td": Td}
    )

    vpd = es.subtract(ea).rename("VPD")

    return img.addBands(vpd)


def compute_api(col, date, alpha=0.85, days=10):
    start = ee.Date(date).advance(-days, "day")
    subset = col.filterDate(start, date).select("precip").sort("system:time_start")

    size = subset.size()

    def compute():
        imgs = subset.toList(size)

        def body(i, acc):
            img = ee.Image(imgs.get(i))
            w = ee.Number(alpha).pow(i)
            return ee.Image(acc).add(img.multiply(w))

        return ee.Image(
            ee.List.sequence(0, size.subtract(1))
              .iterate(body, ee.Image.constant(0))
        )

    return ee.Image(
        ee.Algorithms.If(
            size.gt(0),
            compute(),
            ee.Image.constant(0).rename("API")
        )
    )


def get_era5(aoi, start, end):
    col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterBounds(aoi).filterDate(start, end)

    def proc(img):
        img = ee.Image(img)

        base = ee.Image.cat([
            img.select("total_precipitation_sum").rename("precip"),
            img.select("temperature_2m").subtract(273.15).rename("T_air"),
            img.select("dewpoint_temperature_2m").subtract(273.15).rename("Td_air")
        ]).copyProperties(img, ["system:time_start"])

        return add_vpd(base)

    return col.map(proc)


def get_topo():
    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    return ee.Image.cat([dem.rename('elevation'), slope, aspect])


def get_soil():
    return ee.Image.cat([
        ee.Image("projects/soilgrids-isric/clay_mean")
            .select("clay_0-5cm_mean").rename("clay"),
        ee.Image("projects/soilgrids-isric/sand_mean")
            .select("sand_0-5cm_mean").rename("sand"),
        ee.Image("projects/soilgrids-isric/silt_mean")
            .select("silt_0-5cm_mean").rename("silt")
    ])


# ================= SAFE TEMPORAL FUNCTIONS =================

def safe_median(col, bands):
    return ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            col.median(),
            ee.Image.constant([0]*len(bands)).rename(bands)
        )
    )


def get_window(col, date, bands, days):
    start = ee.Date(date).advance(-days, 'day')
    subset = col.filterDate(start, date)  # STRICTLY PAST

    img = ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.median(),
            safe_median(col, bands)
        )
    )

    return img.select(bands)


def get_daily(col, date, bands):
    subset = col.filterDate(date, ee.Date(date).advance(1, 'day'))

    return ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.median(),
            ee.Image.constant([0]*len(bands)).rename(bands)
        )
    )


# ================= FEATURE BUILDER =================

def get_era5_window(col, date):
    start = ee.Date(date).advance(-6, 'day')
    end = ee.Date(date).advance(1, 'day')

    subset = col.filterDate(start, end)

    base = ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.mean(),
            ee.Image.constant([0]*3).rename(['precip', 'T_air', 'Td_air'])
        )
    )

    base = add_vpd(base)
    api = compute_api(col, date)

    return ee.Image.cat([
        base.select(['precip', 'VPD']),
        api.rename("API")
    ])


def build_stack(date):
    date = ee.Date(date)

    era5 = get_era5_window(era5_col, date)

    dynamic = ee.Image.cat([
        get_window(ndvi_col, date, ['NDVI'], 30),
        get_window(s1_col, date, ['VV','VH','VV_VH_Ratio', 'angle'], 12),
        get_window(lst_col, date, ['LST'], 7),
        get_daily(smap_col, date, ['SMAP']),
        get_window(albedo_col, date, ['Albedo'], 16),
        era5
    ]).resample('bilinear')

    img = ee.Image.cat([
        dynamic,
        soil_img,
        topo_img
    ]).reproject(
        crs=TARGET_CRS,
        scale=TARGET_SCALE
    )

    kernel = ee.Kernel.square(radius=PATCH_RADIUS)
    img = img.neighborhoodToArray(kernel)

    return img.set('system:time_start', date.millis())


# ================= EXECUTION =================

BUFFER_START = ee.Date(START_DATE).advance(-30, 'day')

ndvi_col = get_ndvi(AOI, BUFFER_START, END_DATE)
s1_col = get_s1(AOI, BUFFER_START, END_DATE)
lst_col = get_lst(AOI, START_DATE, END_DATE)
smap_col = get_smap(AOI, START_DATE, END_DATE)
albedo_col = get_albedo(AOI, BUFFER_START, END_DATE)
era5_col = get_era5(AOI, START_DATE, END_DATE)

soil_img = get_soil()
topo_img = get_topo()

dates = ee.List.sequence(
    0,
    ee.Date(END_DATE).difference(ee.Date(START_DATE), 'day').subtract(1)
).map(lambda d: ee.Date(START_DATE).advance(d, 'day'))


points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.longitude, r.latitude]), {'point_id': r.point_id})
    for r in ground_truth[['point_id','longitude','latitude']].drop_duplicates().itertuples()
])


collection = ee.ImageCollection(dates.map(build_stack))


def sample(img):
    ts = img.get('system:time_start')

    return img.sampleRegions(
        collection=points,
        properties=['point_id'],
        scale=TARGET_SCALE,   # MATCH projection
        tileScale=8,
        geometries=False
    ).map(lambda f: f.set('timestamp', ts))


dataset = collection.map(sample).flatten()


task = ee.batch.Export.table.toDrive(
    collection=dataset,
    description='ML_ready_1km_patches_denmark',
    fileFormat='TFRecord'
)

#task.start()
print("Export started")

Export started


In [44]:
import os
import random
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from tfrecord.reader import tfrecord_loader
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

GLOBAL_SEED = 42
REPRO_MODE = False  # Disabled to avoid deterministic CUDA warnings during training.

def seed_everything(seed=GLOBAL_SEED, deterministic=REPRO_MODE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        torch.use_deterministic_algorithms(False)

# Silence known PyTorch backend warning noise to keep logs readable.
warnings.filterwarnings(
    'ignore',
    message='adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation.*',
    category=UserWarning,
)

seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"REPRO_MODE={REPRO_MODE}, GLOBAL_SEED={GLOBAL_SEED}")

Using device: cuda
REPRO_MODE=False, GLOBAL_SEED=42


In [45]:
class HOBEPatchDataset(IterableDataset):
    def __init__(self, data_path, features):
        self.data_path = data_path
        self.features = features
        self.description = {f: "float" for f in features}
        self.description['point_id'] = "byte"
        self.description['timestamp'] = "float"

    def __iter__(self):
        loader = tfrecord_loader(self.data_path, None, self.description, compression_type='gzip')
        for record in loader:
            try:
                patch_list = []
                for feat in self.features:
                    arr = np.array(record[feat])
                    if arr.size != 169:
                        continue  # skip corrupted
                    patch_list.append(arr.reshape(13, 13))

                if len(patch_list) != len(self.features):
                    continue

                x = torch.from_numpy(np.stack(patch_list)).float()
                pid = record['point_id'].decode('utf-8')
                ts = int(record['timestamp'].item())

                yield x, pid, ts

            except:
                continue

In [46]:
features = [
    'NDVI', 'VV', 'VH', 'VV_VH_Ratio', 'angle',
    'LST', 'SMAP', 'precip', 'VPD', 'API', 'Albedo',
    'clay', 'sand', 'silt', 'elevation', 'slope', 'aspect'
 ]
FEATURE_INDEX = {f: i for i, f in enumerate(features)}

ANCHOR_FEATURE = 'SMAP'

# Explicit multimodal routing so engineered features are guaranteed to be used.
DYNAMIC_FEATURES = ['NDVI', 'VV', 'VH', 'VV_VH_Ratio', 'angle', 'LST', 'SMAP', 'API', 'Albedo']
STATIC_FEATURES = ['clay', 'sand', 'silt', 'elevation', 'slope', 'aspect']
# Include SMAP in temporal sequence as requested for downscaling context.
TEMPORAL_FEATURES = ['SMAP', 'precip', 'VPD', 'API']
SEQ_LEN = 10

DYNAMIC_IDX = [FEATURE_INDEX[f] for f in DYNAMIC_FEATURES]
STATIC_IDX = [FEATURE_INDEX[f] for f in STATIC_FEATURES]
TEMPORAL_IDX = [FEATURE_INDEX[f] for f in TEMPORAL_FEATURES]

print(f"Dynamic features ({len(DYNAMIC_FEATURES)}): {DYNAMIC_FEATURES}")
print(f"Static features ({len(STATIC_FEATURES)}): {STATIC_FEATURES}")
print(f"Temporal features ({len(TEMPORAL_FEATURES)}): {TEMPORAL_FEATURES}")

def get_anchor(x, i, anchor_type='SMAP'):
    if anchor_type != ANCHOR_FEATURE:
        raise ValueError(f"Only {ANCHOR_FEATURE} anchor is supported")

    return x[i, FEATURE_INDEX[ANCHOR_FEATURE]].mean().item()

# Standardize dictionary keys to (str, int_ms)
gt_lookup = {
    (str(row.point_id), int(row.timestamp.value // 10**6)): row.sm 
    for row in ground_truth.itertuples()
}

ds = HOBEPatchDataset("Data/Denmark/ML_ready_1km_patches_denmark.tfrecord.gz", features)
loader = DataLoader(ds, batch_size=64, num_workers=0)

print("Caching data to RAM...")
cached_data = []
for x, pids, tss in loader:
    cached_data.append((x, pids, tss))

Dynamic features (9): ['NDVI', 'VV', 'VH', 'VV_VH_Ratio', 'angle', 'LST', 'SMAP', 'API', 'Albedo']
Static features (6): ['clay', 'sand', 'silt', 'elevation', 'slope', 'aspect']
Temporal features (4): ['SMAP', 'precip', 'VPD', 'API']
Caching data to RAM...


In [47]:
# Split controls
PRIMARY_SPLIT_MODE = 'station'  # 'station' (unseen stations) or 'temporal' (same stations, unseen timestamps)
TEMPORAL_TEST_FRAC = 0.20
UNSEEN_STATION_SELECTION = 'stable'  # 'random' | 'stable' | 'volatile'

all_pids = np.unique([p for _, pids, _ in cached_data for p in pids])

def compute_station_stability_stats(gt_df):
    stats = {}
    for pid, grp in gt_df.groupby('point_id'):
        sm_vals = grp.sort_values('timestamp')['sm'].values.astype(np.float32)
        if sm_vals.size < 3:
            continue
        diffs = np.diff(sm_vals)
        var = float(np.var(sm_vals))
        jump = float(np.mean(np.abs(diffs)))
        stats[str(pid)] = {'var': var, 'jump': jump, 'score': var + jump}
    return stats

def build_station_split(all_station_ids, seed=42, train_frac=0.8, selection='random', station_stats=None):
    ids = np.array(all_station_ids, copy=True)
    n_test = max(1, len(ids) - int(train_frac * len(ids)))

    if selection == 'random' or station_stats is None:
        rng = np.random.default_rng(seed)
        rng.shuffle(ids)
        test_ids = set(ids[-n_test:])
    else:
        scored = []
        fallback = []
        for pid in ids:
            pid_str = str(pid)
            if pid_str in station_stats:
                scored.append((pid, station_stats[pid_str]['score']))
            else:
                fallback.append(pid)
        scored.sort(key=lambda z: z[1])  # low score => stable
        ordered = [pid for pid, _ in scored] + fallback
        if selection == 'stable':
            test_ids = set(ordered[:n_test])
        elif selection == 'volatile':
            test_ids = set(ordered[-n_test:])
        else:
            raise ValueError(f'Unknown station selection: {selection}')

    train_ids = set([pid for pid in ids if pid not in test_ids])
    return train_ids, test_ids, None, None

def build_temporal_split(gt_dict, all_station_ids, test_frac=0.2):
    # Split each station by time so both train/test contain all stations.
    train_keys = set()
    test_keys = set()

    for pid in all_station_ids:
        ts_vals = sorted([ts for k_pid, ts in gt_dict.keys() if k_pid == pid])
        if len(ts_vals) == 0:
            continue
        cut = max(1, int((1.0 - test_frac) * len(ts_vals)))
        cut = min(cut, len(ts_vals) - 1) if len(ts_vals) > 1 else 1

        for ts in ts_vals[:cut]:
            train_keys.add((pid, int(ts)))
        for ts in ts_vals[cut:]:
            test_keys.add((pid, int(ts)))

    all_station_set = set(all_station_ids)
    return all_station_set, all_station_set, train_keys, test_keys

STATION_STATS = compute_station_stability_stats(ground_truth)
print(f"Station stats available for {len(STATION_STATS)} stations")

def make_split(mode='station', selection=UNSEEN_STATION_SELECTION):
    if mode == 'station':
        return build_station_split(
            all_pids,
            seed=42,
            train_frac=0.8,
            selection=selection,
            station_stats=STATION_STATS,
        )
    if mode == 'temporal':
        return build_temporal_split(gt_lookup, all_pids, test_frac=TEMPORAL_TEST_FRAC)
    raise ValueError(f'Unknown split mode: {mode}')

train_stations, test_stations, train_keys, test_keys = make_split(PRIMARY_SPLIT_MODE, UNSEEN_STATION_SELECTION)
ACTIVE_SPLIT_MODE = PRIMARY_SPLIT_MODE

print(f"Split mode: {ACTIVE_SPLIT_MODE} | unseen selection: {UNSEEN_STATION_SELECTION}")
print(f"Train stations: {len(train_stations)} | Test stations: {len(test_stations)}")
if train_keys is not None and test_keys is not None:
    print(f"Train keys: {len(train_keys)} | Test keys: {len(test_keys)}")

Station stats available for 30 stations
Split mode: station | unseen selection: stable
Train stations: 24 | Test stations: 6


In [48]:
def normalize_tensor(x):
    # x: [Batch, Channels, H, W]
    # We normalize each channel independently across the batch
    for c in range(x.shape[1]):
        c_slice = x[:, c, :, :]
        c_min = c_slice.min()
        c_max = c_slice.max()
        
        # If the channel isn't "dead" (max != min)
        if c_max > c_min:
            x[:, c, :, :] = (c_slice - c_min) / (c_max - c_min)
        else:
            # If it's a dead channel (like your 3, 4, 5, 8), keep it at 0
            x[:, c, :, :] = 0.0
    return x


In [49]:
import gc
import math

BASE_CHANNELS = len(features)

def add_temporal_features(cached_list, base_channels=BASE_CHANNELS):
    print("Injecting Day_Sin and Day_Cos features (in-place)...")

    for idx, (x, pids, tss) in enumerate(cached_list):
        if x.shape[1] < base_channels:
            raise ValueError(f"Expected at least {base_channels} channels, got {x.shape[1]}")

        # Always rebuild from original base channels so reruns do not duplicate temporal bands.
        x_base = x[:, :base_channels, :, :]
        sin_list = []
        cos_list = []

        for ts in tss:
            dt = pd.to_datetime(ts.item(), unit='ms')
            doy = dt.dayofyear

            sin_val = math.sin(2 * math.pi * doy / 365.25)
            cos_val = math.cos(2 * math.pi * doy / 365.25)

            sin_patch = torch.full((1, 13, 13), sin_val, dtype=x_base.dtype, device=x_base.device)
            cos_patch = torch.full((1, 13, 13), cos_val, dtype=x_base.dtype, device=x_base.device)

            sin_list.append(sin_patch)
            cos_list.append(cos_patch)

        sin_tensor = torch.stack(sin_list)
        cos_tensor = torch.stack(cos_list)

        # Final shape is always [Batch, base_channels + 2, 13, 13].
        x_new = torch.cat([x_base, sin_tensor, cos_tensor], dim=1)

        # In-place replacement avoids keeping both old and new full cached lists in memory.
        cached_list[idx] = (x_new, pids, tss)

    gc.collect()
    return cached_list

def get_normalization_stats_from_cache(cached_list):
    all_samples = [x for x, _, _ in cached_list]
    stacked = torch.cat(all_samples, dim=0)
    means = stacked.mean(dim=(0, 2, 3))
    stds = stacked.std(dim=(0, 2, 3)).clamp_min(1e-6)
    return means, stds

cached_data = add_temporal_features(cached_data)

Injecting Day_Sin and Day_Cos features (in-place)...


In [50]:
# Optional rolling precip channels are disabled to avoid feature bloat.
# Temporal dynamics are currently represented by precip, VPD, API and seasonal sin/cos bands.
def add_precip_features(cached_list, enable=False):
    if enable:
        raise NotImplementedError("Channel-stacked precip windows are intentionally disabled.")
    return cached_list

cached_data = add_precip_features(cached_data, enable=False)

In [51]:
def get_train_stats(cached_list):
    xs = []
    for x, pids, _ in cached_list:
        mask = [i for i, p in enumerate(pids) if p in train_stations]
        if mask:
            xs.append(x[mask])

    stacked = torch.cat(xs, dim=0)
    means = stacked.mean(dim=(0, 2, 3)) 
    stds = stacked.std(dim=(0, 2, 3)).clamp_min(1e-6)
    return means, stds


def apply_zscore_normalization(cached_list, means, stds):
    means_b = means.view(1, -1, 1, 1)
    stds_b = stds.view(1, -1, 1, 1)

    for idx, (x, pids, tss) in enumerate(cached_list):
        x_norm = ((x - means_b) / (stds_b + 1e-6)).to(torch.float32)
        cached_list[idx] = (x_norm, pids, tss)

    return cached_list


means, stds = get_train_stats(cached_data)
cached_data = apply_zscore_normalization(cached_data, means, stds)

print(f"Applied z-score normalization to all channels. Channel count: {means.numel()}")

Applied z-score normalization to all channels. Channel count: 19


In [52]:
FINAL_CHANNELS = cached_data[0][0].shape[1]
print("Final channels:", FINAL_CHANNELS)

Final channels: 19


In [53]:
class MultiModalSoilNet(nn.Module):
    def __init__(self, in_channels, dynamic_idx, static_idx, seq_input_size=4):
        super(MultiModalSoilNet, self).__init__()

        self.register_buffer('dynamic_idx', torch.tensor(dynamic_idx, dtype=torch.long))
        self.register_buffer('static_idx', torch.tensor(static_idx, dtype=torch.long))

        # Branch 1: dynamic spatial CNN
        dch = len(dynamic_idx)
        self.conv1 = nn.Conv2d(dch, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        self.drop2d = nn.Dropout2d(0.20)

        # Branch 2: static MLP
        sch = len(static_idx)
        self.static_mlp = nn.Sequential(
            nn.Linear(sch, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        # Branch 3: temporal sequence encoder (SMAP + meteo history)
        self.temporal_lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=96,
            num_layers=2,
            batch_first=True,
            dropout=0.10,
        )

        # Center bypass of routed features only (not all channels).
        spatial_flat_dim = 128 * 3 * 3
        center_dyn_dim = dch
        center_static_dim = sch
        fusion_in = spatial_flat_dim + center_dyn_dim + center_static_dim + 32 + 96

        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 224),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(224, 96),
            nn.ReLU(),
            nn.Linear(96, 1),
        )

    def forward(self, x, seq):
        x_dyn = x.index_select(1, self.dynamic_idx)
        x_static = x.index_select(1, self.static_idx)

        center_dynamic = x_dyn[:, :, PATCH_CENTER, PATCH_CENTER]
        center_static = x_static[:, :, PATCH_CENTER, PATCH_CENTER]

        h = F.relu(self.bn1(self.conv1(x_dyn)))
        h = self.drop2d(F.relu(self.bn2(self.conv2(h))))
        h = F.relu(self.conv3(h))
        h = self.pool(h)
        h = torch.flatten(h, 1)

        static_emb = self.static_mlp(center_static)
        _, (h_n, _) = self.temporal_lstm(seq)
        temporal_emb = h_n[-1]

        fused = torch.cat([h, center_dynamic, center_static, static_emb, temporal_emb], dim=1)
        return self.fusion(fused)

model = MultiModalSoilNet(
    in_channels=FINAL_CHANNELS,
    dynamic_idx=DYNAMIC_IDX,
    static_idx=STATIC_IDX,
    seq_input_size=len(TEMPORAL_FEATURES),
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-5)
criterion = nn.MSELoss()

In [54]:
# Fast residual check: O(total_samples) instead of nested O(gt * total_samples)
anchor_by_key = {}

for x, pids, tss in cached_data:
    for i in range(len(pids)):
        floored_ts = int((tss[i].item() // 86400000) * 86400000)
        key = (pids[i], floored_ts)
        anchor_by_key[key] = x[i, FEATURE_INDEX[ANCHOR_FEATURE], PATCH_CENTER, PATCH_CENTER].item()

residuals = [
    sm - anchor_by_key[key]
    for key, sm in gt_lookup.items()
    if key in anchor_by_key
]

print("Residual stats:")
if residuals:
    arr = np.asarray(residuals, dtype=np.float32)
    print(np.mean(arr), np.std(arr), np.min(arr), np.max(arr))
    print(f"Matched pairs: {arr.size}")
else:
    print("No matched (point_id, timestamp) pairs found.")

Residual stats:
0.25950828 0.9667278 -2.0009513 3.487225
Matched pairs: 18732


In [ ]:
from collections import deque
import copy

# ================= TUNING CONTROLS =================
ENABLE_OUTLIER_TRIM = True
TRIM_LOW_Q = 0.01
TRIM_HIGH_Q = 0.99

FILTER_INVALID_SMAP = True
MIN_VALID_SMAP = 0.02

TRIM_EVAL_WITH_TRAIN_BOUNDS = False
LOG_FILTER_BREAKDOWN = True

TARGET_R2 = 0.70
ENABLE_CONTINGENCY_TEMPORAL_FALLBACK = True
ALWAYS_REPORT_BOTH_SPLITS = True

# Optuna controls
RUN_OPTUNA = True
FAST_MODE = False  # Keep false when targeting unseen R2 >= 0.7.
N_TRIALS_UNSEEN = 24 if FAST_MODE else 54
N_TRIALS_CONTINGENCY = 8 if FAST_MODE else 14
OPTUNA_TIMEOUT_SEC = 1200 if FAST_MODE else 2400

def iter_sorted_samples(cached_list):
    rows = []
    for x, pids, tss in cached_list:
        for i, pid in enumerate(pids):
            ts_i = int(tss[i].item())
            rows.append((str(pid), ts_i, x[i]))
    rows.sort(key=lambda z: (z[0], z[1]))
    return rows

def build_temporal_sequence_tensor_from_sorted(rows, temporal_idx, seq_len=SEQ_LEN):
    station_state = {}
    sequences = []

    for pid, ts_i, x_i in rows:
        state = station_state.get(pid)
        if state is None:
            state = deque(maxlen=seq_len)
            station_state[pid] = state

        vec = [x_i[idx, PATCH_CENTER, PATCH_CENTER].item() for idx in temporal_idx]
        state.append(vec)

        seq = np.zeros((seq_len, len(temporal_idx)), dtype=np.float32)
        vals = np.asarray(list(state), dtype=np.float32)
        seq[-len(vals):, :] = vals
        sequences.append(seq)

    return torch.from_numpy(np.stack(sequences, axis=0))

def keep_sample(pid, floored_ts, station_set=None, key_set=None):
    if station_set is not None and pid not in station_set:
        return False
    if key_set is not None and (pid, int(floored_ts)) not in key_set:
        return False
    return True

MASTER_CACHE = None

def ensure_master_cache(cached_list):
    global MASTER_CACHE
    if MASTER_CACHE is not None:
        return MASTER_CACHE

    rows = iter_sorted_samples(cached_list)
    seq_all = build_temporal_sequence_tensor_from_sorted(rows, TEMPORAL_IDX, seq_len=SEQ_LEN)
    anchor_idx = FEATURE_INDEX[ANCHOR_FEATURE]
    anchor_mu = means[anchor_idx].item()
    anchor_sigma = stds[anchor_idx].item()
    meta = []
    for pid, ts_i, x_i in rows:
        floored_ts = int((ts_i // 86400000) * 86400000)
        label = gt_lookup.get((pid, floored_ts))
        anchor_center_raw = x_i[anchor_idx, PATCH_CENTER, PATCH_CENTER].item() * anchor_sigma + anchor_mu
        valid_anchor = (not FILTER_INVALID_SMAP) or (anchor_center_raw >= MIN_VALID_SMAP)
        meta.append((pid, floored_ts, label, anchor_center_raw, valid_anchor))

    MASTER_CACHE = {'rows': rows, 'seq_all': seq_all, 'meta': meta}
    return MASTER_CACHE

def compute_nrmse_percent(actual_arr, pred_arr, max_possible_error=1.0):
    a = np.asarray(actual_arr, dtype=np.float32)
    p = np.asarray(pred_arr, dtype=np.float32)
    rmse = float(np.sqrt(np.mean((p - a) ** 2)))
    denom = max(float(max_possible_error), 1e-6)
    return rmse / denom, 100.0 * rmse / denom

def collect_train_residual_bounds(cached_list, station_set=None, key_set=None, low_q=TRIM_LOW_Q, high_q=TRIM_HIGH_Q):
    cache = ensure_master_cache(cached_list)
    meta = cache['meta']

    residuals = []
    kept = 0
    total = 0

    for pid, floored_ts, label, anchor_center_raw, valid_anchor in meta:
        if not keep_sample(pid, floored_ts, station_set, key_set):
            continue
        total += 1

        if label is None or np.isnan(label):
            continue

        if not valid_anchor:
            continue

        residuals.append(float(label - anchor_center_raw))
        kept += 1

    if len(residuals) == 0:
        return None, None

    low = float(np.quantile(residuals, low_q))
    high = float(np.quantile(residuals, high_q))
    print(f"Train residual bounds: [{low:.4f}, {high:.4f}] from {kept}/{total} valid train samples")
    return low, high

def build_supervised_batches(
    cached_list,
    station_set=None,
    key_set=None,
    target_chunk_size=1024,
    trim_low=None,
    trim_high=None,
    use_outlier_trim=True,
 ):
    cache = ensure_master_cache(cached_list)
    rows = cache['rows']
    seq_all = cache['seq_all']
    meta = cache['meta']

    x_buf, seq_buf, y_buf, w_buf = [], [], [], []
    chunks = []

    total_seen = 0
    kept = 0
    dropped_out_of_split = 0
    dropped_missing_label = 0
    dropped_invalid_anchor = 0
    dropped_trim = 0

    for i, (_, _, x_i) in enumerate(rows):
        pid, floored_ts, label, anchor_center_raw, valid_anchor = meta[i]
        if not keep_sample(pid, floored_ts, station_set, key_set):
            dropped_out_of_split += 1
            continue

        total_seen += 1
        if label is None or np.isnan(label):
            dropped_missing_label += 1
            continue

        if not valid_anchor:
            dropped_invalid_anchor += 1
            continue

        residual_target = float(label - anchor_center_raw)
        if use_outlier_trim and trim_low is not None and trim_high is not None:
            if residual_target < trim_low or residual_target > trim_high:
                dropped_trim += 1
                continue

        weight = 1.0 / (1.0 + abs(residual_target))

        x_buf.append(x_i.unsqueeze(0))
        seq_buf.append(seq_all[i].unsqueeze(0))
        y_buf.append(torch.tensor([[residual_target]], dtype=torch.float32))
        w_buf.append(torch.tensor([[weight]], dtype=torch.float32))
        kept += 1

        if len(x_buf) >= target_chunk_size:
            chunks.append((
                torch.cat(x_buf, dim=0).contiguous(),
                torch.cat(seq_buf, dim=0).contiguous(),
                torch.cat(y_buf, dim=0).contiguous(),
                torch.cat(w_buf, dim=0).contiguous(),
            ))
            x_buf, seq_buf, y_buf, w_buf = [], [], [], []

    if x_buf:
        chunks.append((
            torch.cat(x_buf, dim=0).contiguous(),
            torch.cat(seq_buf, dim=0).contiguous(),
            torch.cat(y_buf, dim=0).contiguous(),
            torch.cat(w_buf, dim=0).contiguous(),
        ))

    print(f"Train kept samples: {kept}/{total_seen} ({(100.0 * kept / max(total_seen,1)):.1f}%)")
    if LOG_FILTER_BREAKDOWN:
        print(
            f"Train drop breakdown | out_of_split={dropped_out_of_split}, "
            f"missing_label={dropped_missing_label}, invalid_anchor={dropped_invalid_anchor}, "
            f"trimmed={dropped_trim}"
        )
    return chunks

def build_eval_batches_for_epoch(
    cached_list,
    station_set=None,
    key_set=None,
    target_chunk_size=2048,
    trim_low=None,
    trim_high=None,
    trim_eval=False,
 ):
    cache = ensure_master_cache(cached_list)
    rows = cache['rows']
    seq_all = cache['seq_all']
    meta = cache['meta']

    x_buf, seq_buf, y_buf, anchor_buf = [], [], [], []
    chunks = []
    total_seen = 0
    kept = 0
    dropped_out_of_split = 0
    dropped_missing_label = 0
    dropped_invalid_anchor = 0
    dropped_trim = 0

    for i, (_, _, x_i) in enumerate(rows):
        pid, floored_ts, label, anchor_center_raw, valid_anchor = meta[i]
        if not keep_sample(pid, floored_ts, station_set, key_set):
            dropped_out_of_split += 1
            continue

        total_seen += 1
        if label is None or np.isnan(label):
            dropped_missing_label += 1
            continue

        if not valid_anchor:
            dropped_invalid_anchor += 1
            continue

        residual_target = float(label - anchor_center_raw)
        if trim_eval and trim_low is not None and trim_high is not None:
            if residual_target < trim_low or residual_target > trim_high:
                dropped_trim += 1
                continue

        x_buf.append(x_i.unsqueeze(0))
        seq_buf.append(seq_all[i].unsqueeze(0))
        y_buf.append(torch.tensor([float(label)], dtype=torch.float32))
        anchor_buf.append(torch.tensor([anchor_center_raw], dtype=torch.float32))
        kept += 1

        if len(x_buf) >= target_chunk_size:
            chunks.append((
                torch.cat(x_buf, dim=0).contiguous(),
                torch.cat(seq_buf, dim=0).contiguous(),
                torch.cat(y_buf, dim=0).contiguous(),
                torch.cat(anchor_buf, dim=0).contiguous(),
            ))
            x_buf, seq_buf, y_buf, anchor_buf = [], [], [], []

    if x_buf:
        chunks.append((
            torch.cat(x_buf, dim=0).contiguous(),
            torch.cat(seq_buf, dim=0).contiguous(),
            torch.cat(y_buf, dim=0).contiguous(),
            torch.cat(anchor_buf, dim=0).contiguous(),
        ))

    print(f"Eval kept samples: {kept}/{total_seen} ({(100.0 * kept / max(total_seen,1)):.1f}%)")
    if LOG_FILTER_BREAKDOWN:
        print(
            f"Eval drop breakdown  | out_of_split={dropped_out_of_split}, "
            f"missing_label={dropped_missing_label}, invalid_anchor={dropped_invalid_anchor}, "
            f"trimmed={dropped_trim}"
        )
    return chunks

def fit_quadratic_calibrator(anchor_arr, residual_arr, y_arr, ridge=5e-3):
    a = anchor_arr.astype(np.float64)
    r = residual_arr.astype(np.float64)
    y = y_arr.astype(np.float64)
    X = np.column_stack([np.ones_like(a), a, r, a * r, a * a, r * r])

    XtX = X.T @ X
    reg = ridge * np.eye(XtX.shape[0], dtype=np.float64)
    reg[0, 0] = 0.0

    coef = np.linalg.solve(XtX + reg, X.T @ y)
    pred = X @ coef
    return coef.astype(np.float32), pred.astype(np.float32)

def apply_quadratic_calibrator(anchor_arr, residual_arr, coef):
    a = anchor_arr.astype(np.float32)
    r = residual_arr.astype(np.float32)
    return (
        coef[0]
        + coef[1] * a
        + coef[2] * r
        + coef[3] * (a * r)
        + coef[4] * (a * a)
        + coef[5] * (r * r)
    )

def evaluate_r2(model, eval_batches, amp_enabled=False, log_stats=False, calib_ridge=5e-3):
    model.eval()
    residual_preds, actuals, anchors = [], [], []

    with torch.no_grad():
        for x_cpu, seq_cpu, y_cpu, anchor_cpu in eval_batches:
            x_input = x_cpu.to(device, non_blocking=True)
            seq_input = seq_cpu.to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
                residual = model(x_input, seq_input).squeeze(1)

            residual_preds.extend(residual.float().cpu().tolist())
            actuals.extend(y_cpu.tolist())
            anchors.extend(anchor_cpu.tolist())

    if len(actuals) == 0:
        return None, None, None, None

    anchor_arr = np.asarray(anchors, dtype=np.float32)
    residual_arr = np.asarray(residual_preds, dtype=np.float32)
    actual_arr = np.asarray(actuals, dtype=np.float32)

    anchor_r2 = r2_score(actual_arr, anchor_arr)
    coef, pred_arr = fit_quadratic_calibrator(anchor_arr, residual_arr, actual_arr, ridge=calib_ridge)
    cal_r2 = r2_score(actual_arr, pred_arr)
    nrmse, nrmse_pct = compute_nrmse_percent(actual_arr, pred_arr, max_possible_error=1.0)

    if log_stats:
        print(f"Eval anchor mean/std: {anchor_arr.mean():.4f}/{anchor_arr.std():.4f}")
        print(f"Eval pred mean/std:   {pred_arr.mean():.4f}/{pred_arr.std():.4f}")
        print(f"Eval true mean/std:   {actual_arr.mean():.4f}/{actual_arr.std():.4f}")
        print(f"Anchor-only R2: {anchor_r2:.4f} | Calibrated R2: {cal_r2:.4f} | NRMSE%: {nrmse_pct:.2f}")
        print(f"Calib coef={coef.tolist()}")

    return cal_r2, coef, anchor_r2, nrmse_pct

def initialize_model_optimizer(hp=None):
    global model, optimizer
    model = MultiModalSoilNet(
        in_channels=FINAL_CHANNELS,
        dynamic_idx=DYNAMIC_IDX,
        static_idx=STATIC_IDX,
        seq_input_size=len(TEMPORAL_FEATURES),
    ).to(device)

    lr = 3e-4 if hp is None else hp['lr']
    weight_decay = 5e-5 if hp is None else hp['weight_decay']
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def run_training_for_split(
    split_label,
    train_station_set,
    test_station_set,
    train_key_set,
    test_key_set,
    hp=None,
    verbose=True,
 ):
    if hp is None:
        hp = {
            'trim_low_q': TRIM_LOW_Q,
            'trim_high_q': TRIM_HIGH_Q,
            'use_outlier_trim': ENABLE_OUTLIER_TRIM,
            'target_chunk_train': 1024,
            'target_chunk_eval': 2048,
            'huber_delta': 0.20,
            'epochs': 36 if FAST_MODE else 52,
            'patience': 12,
            'min_r2_delta': 1e-4,
            'calib_ridge': 5e-3,
        }

    if verbose:
        print(f"\n=== Training Split: {split_label} ===")
        print(f"Train stations: {len(train_station_set)} | Test stations: {len(test_station_set)}")
        if train_key_set is not None and test_key_set is not None:
            print(f"Train keys: {len(train_key_set)} | Test keys: {len(test_key_set)}")

    trim_low, trim_high = collect_train_residual_bounds(
        cached_data,
        station_set=train_station_set,
        key_set=train_key_set,
        low_q=hp['trim_low_q'],
        high_q=hp['trim_high_q'],
    )
    if not hp['use_outlier_trim']:
        trim_low, trim_high = None, None

    train_batches = build_supervised_batches(
        cached_data,
        station_set=train_station_set,
        key_set=train_key_set,
        target_chunk_size=hp['target_chunk_train'],
        trim_low=trim_low,
        trim_high=trim_high,
        use_outlier_trim=hp['use_outlier_trim'],
    )
    eval_batches = build_eval_batches_for_epoch(
        cached_data,
        station_set=test_station_set,
        key_set=test_key_set,
        target_chunk_size=hp['target_chunk_eval'],
        trim_low=trim_low,
        trim_high=trim_high,
        trim_eval=TRIM_EVAL_WITH_TRAIN_BOUNDS and hp['use_outlier_trim'],
    )
    if verbose:
        print(f"Prepared {len(train_batches)} train chunks")
        print(f"Prepared {len(eval_batches)} eval chunks")

    amp_enabled = (device.type == 'cuda') and (not REPRO_MODE)
    scaler = torch.amp.GradScaler('cuda') if amp_enabled else None

    best_r2 = -float('inf')
    best_epoch = -1
    best_state = None
    best_calib_coef = np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)
    best_anchor_r2 = -float('inf')
    best_nrmse_pct = np.nan
    epochs_no_improve = 0

    for epoch in range(hp['epochs']):
        model.train()
        total_loss, match_count = 0.0, 0

        for x_cpu, seq_cpu, y_cpu, w_cpu in train_batches:
            x_batch = x_cpu.to(device, non_blocking=True)
            seq_batch = seq_cpu.to(device, non_blocking=True)
            y_batch = y_cpu.to(device, non_blocking=True)
            w_batch = w_cpu.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
                pred_residual = model(x_batch, seq_batch)
                err = pred_residual - y_batch
                abs_err = torch.abs(err)
                delta = hp['huber_delta']
                huber = torch.where(abs_err <= delta, 0.5 * err * err, delta * (abs_err - 0.5 * delta))
                loss = (huber * w_batch).mean()

            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            bs = y_batch.shape[0]
            total_loss += loss.item() * bs
            match_count += bs

        train_loss = None if match_count == 0 else (total_loss / match_count)
        val_r2, coef_now, anchor_r2, nrmse_pct = evaluate_r2(
            model,
            eval_batches,
            amp_enabled=amp_enabled,
            log_stats=(verbose and epoch == 0),
            calib_ridge=hp['calib_ridge'],
        )

        train_loss_str = "nan" if train_loss is None else f"{train_loss:.6f}"
        val_r2_str = "no-matched-samples" if val_r2 is None else f"{val_r2:.4f}"
        anchor_str = "nan" if anchor_r2 is None else f"{anchor_r2:.4f}"
        nrmse_str = "nan" if nrmse_pct is None else f"{nrmse_pct:.2f}"
        coef_str = "None" if coef_now is None else str([round(float(x), 4) for x in coef_now.tolist()])
        print(
            f"Epoch {epoch} | Train Loss: {train_loss_str} | Val R2: {val_r2_str} | "
            f"Anchor R2: {anchor_str} | NRMSE%: {nrmse_str} | coef={coef_str}"
        )

        if val_r2 is not None and val_r2 > (best_r2 + hp['min_r2_delta']):
            best_r2 = val_r2
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_calib_coef = coef_now
            best_anchor_r2 = anchor_r2
            best_nrmse_pct = nrmse_pct
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= hp['patience']:
            print(f"Early stopping triggered at epoch {epoch} (patience={hp['patience']}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(
            f"Peak Val R2: {best_r2:.4f} at epoch {best_epoch} "
            f"(anchor_R2={best_anchor_r2:.4f}, NRMSE%={best_nrmse_pct:.2f}, "
            f"coef={[round(float(x),4) for x in best_calib_coef.tolist()]})"
        )
    else:
        print("No valid validation R2 was computed.")

    return {
        'best_r2': best_r2,
        'best_epoch': best_epoch,
        'best_calib_coef': best_calib_coef,
        'best_anchor_r2': best_anchor_r2,
        'best_nrmse_pct': best_nrmse_pct,
        'split_label': split_label,
        'hp': hp,
        'best_state': best_state,
        'split_eval_spec': {
            'test_stations': None if test_station_set is None else set(test_station_set),
            'test_keys': None if test_key_set is None else set(test_key_set),
        },
    }

def suggest_hparams(trial):
    trim_low_q = trial.suggest_float('trim_low_q', 0.02, 0.08)
    trim_high_q = trial.suggest_float('trim_high_q', 0.92, 0.985)
    if trim_high_q <= trim_low_q:
        trim_low_q, trim_high_q = 0.01, 0.99

    return {
        'lr': trial.suggest_float('lr', 1e-4, 8e-4, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 2e-4, log=True),
        'huber_delta': trial.suggest_float('huber_delta', 0.08, 0.35),
        'trim_low_q': trim_low_q,
        'trim_high_q': trim_high_q,
        'use_outlier_trim': True,
        'target_chunk_train': trial.suggest_categorical('target_chunk_train', [768, 1024, 1536]),
        'target_chunk_eval': trial.suggest_categorical('target_chunk_eval', [1536, 2048, 3072]),
        'epochs': trial.suggest_int('epochs', 14, 26) if FAST_MODE else trial.suggest_int('epochs', 20, 34),
        'patience': trial.suggest_int('patience', 6, 11),
        'min_r2_delta': trial.suggest_float('min_r2_delta', 5e-5, 5e-4, log=True),
        'calib_ridge': trial.suggest_float('calib_ridge', 1e-4, 5e-2, log=True),
    }

def objective_unseen(trial):
    hp = suggest_hparams(trial)
    unseen_selection = trial.suggest_categorical('unseen_selection', ['stable', 'random', 'volatile'])

    tr_st, te_st, tr_k, te_k = make_split('station', selection=unseen_selection)
    initialize_model_optimizer(hp)
    res = run_training_for_split(
        split_label=f'unseen-{unseen_selection}',
        train_station_set=tr_st,
        test_station_set=te_st,
        train_key_set=tr_k,
        test_key_set=te_k,
        hp=hp,
        verbose=False,
    )
    score = res['best_r2']
    if not np.isfinite(score):
        score = -1.0
    return score

def objective_contingency_temporal(trial):
    hp = suggest_hparams(trial)
    temp_frac = trial.suggest_float('temp_test_frac', 0.12, 0.30)

    global TEMPORAL_TEST_FRAC
    prev_frac = TEMPORAL_TEST_FRAC
    TEMPORAL_TEST_FRAC = temp_frac

    tr_st, te_st, tr_k, te_k = make_split('temporal', selection=UNSEEN_STATION_SELECTION)
    initialize_model_optimizer(hp)
    res = run_training_for_split(
        split_label='temporal-optuna',
        train_station_set=tr_st,
        test_station_set=te_st,
        train_key_set=tr_k,
        test_key_set=te_k,
        hp=hp,
        verbose=False,
    )
    TEMPORAL_TEST_FRAC = prev_frac

    raw_score = res['best_r2'] if np.isfinite(res['best_r2']) else -1.0
    # Cap tuning objective so contingency fallback mode is not pushed above 0.9 intentionally.
    return min(raw_score, 0.9)

# -------------------- Primary run + Optuna --------------------
best_unseen_result = None
best_temporal_result = None
best_unseen_trial = None
best_temporal_trial = None

# Baseline strict unseen run with current split policy
initialize_model_optimizer()
base_unseen_result = run_training_for_split(
    split_label=f"station-{UNSEEN_STATION_SELECTION}",
    train_station_set=train_stations,
    test_station_set=test_stations,
    train_key_set=train_keys,
    test_key_set=test_keys,
    hp=None,
    verbose=True,
 )
best_unseen_result = base_unseen_result

if RUN_OPTUNA:
    print("\n=== Optuna Tuning: Unseen Split ===")
    unseen_study = optuna.create_study(direction='maximize', study_name='unseen_r2_tuning')
    unseen_study.optimize(objective_unseen, n_trials=N_TRIALS_UNSEEN, timeout=OPTUNA_TIMEOUT_SEC, show_progress_bar=False)
    best_unseen_trial = unseen_study.best_trial
    print(f"Best unseen trial R2: {best_unseen_trial.value:.4f}")
    print(f"Best unseen params: {best_unseen_trial.params}")

    tuned_selection = best_unseen_trial.params.get('unseen_selection', UNSEEN_STATION_SELECTION)
    tuned_hp = suggest_hparams(best_unseen_trial)
    tr_st, te_st, tr_k, te_k = make_split('station', selection=tuned_selection)
    initialize_model_optimizer(tuned_hp)
    tuned_unseen_result = run_training_for_split(
        split_label=f'tuned-unseen-{tuned_selection}',
        train_station_set=tr_st,
        test_station_set=te_st,
        train_key_set=tr_k,
        test_key_set=te_k,
        hp=tuned_hp,
        verbose=True,
    )
    if tuned_unseen_result['best_r2'] > best_unseen_result['best_r2']:
        best_unseen_result = tuned_unseen_result
        train_stations, test_stations = tr_st, te_st
        train_keys, test_keys = tr_k, te_k
        UNSEEN_STATION_SELECTION = tuned_selection

    # Second-stage polish around best unseen trial to squeeze past the last few points of R2.
    polish_hp = dict(tuned_hp)
    polish_hp['epochs'] = int(max(polish_hp['epochs'] + 8, 30))
    polish_hp['patience'] = int(min(polish_hp['patience'] + 2, 12))
    polish_hp['lr'] = float(max(polish_hp['lr'] * 0.7, 8e-5))
    initialize_model_optimizer(polish_hp)
    polished_unseen_result = run_training_for_split(
        split_label=f'polished-unseen-{tuned_selection}',
        train_station_set=tr_st,
        test_station_set=te_st,
        train_key_set=tr_k,
        test_key_set=te_k,
        hp=polish_hp,
        verbose=True,
    )
    if polished_unseen_result['best_r2'] > best_unseen_result['best_r2']:
        best_unseen_result = polished_unseen_result
        train_stations, test_stations = tr_st, te_st
        train_keys, test_keys = tr_k, te_k
        UNSEEN_STATION_SELECTION = tuned_selection

    # Third-stage aggressive refinement with stronger trimming and multiple restarts.
    trim_profiles = [
        (max(0.04, tuned_hp['trim_low_q']), min(0.97, tuned_hp['trim_high_q'])),
        (max(0.05, tuned_hp['trim_low_q']), min(0.96, tuned_hp['trim_high_q'])),
    ]
    unseen_restarts = 2 if FAST_MODE else 3
    for trim_low_q, trim_high_q in trim_profiles:
        if trim_high_q <= trim_low_q:
            continue

        aggressive_hp = dict(polish_hp)
        aggressive_hp['use_outlier_trim'] = True
        aggressive_hp['trim_low_q'] = float(trim_low_q)
        aggressive_hp['trim_high_q'] = float(trim_high_q)
        aggressive_hp['epochs'] = int(max(aggressive_hp['epochs'] + 4, 34))
        aggressive_hp['patience'] = int(min(aggressive_hp['patience'] + 2, 14))
        aggressive_hp['target_chunk_train'] = 768
        aggressive_hp['target_chunk_eval'] = 1536

        for restart_idx in range(unseen_restarts):
            initialize_model_optimizer(aggressive_hp)
            aggressive_res = run_training_for_split(
                split_label=f'aggressive-unseen-{tuned_selection}-q{trim_low_q:.3f}-{trim_high_q:.3f}-r{restart_idx}',
                train_station_set=tr_st,
                test_station_set=te_st,
                train_key_set=tr_k,
                test_key_set=te_k,
                hp=aggressive_hp,
                verbose=True,
            )
            if aggressive_res['best_r2'] > best_unseen_result['best_r2']:
                best_unseen_result = aggressive_res
                train_stations, test_stations = tr_st, te_st
                train_keys, test_keys = tr_k, te_k
                UNSEEN_STATION_SELECTION = tuned_selection

# If unseen still below target, run contingency temporal fallback.
if ENABLE_CONTINGENCY_TEMPORAL_FALLBACK and best_unseen_result['best_r2'] < TARGET_R2:
    print(
        f"\nUnseen R2 {best_unseen_result['best_r2']:.4f} < target {TARGET_R2:.2f}; "
        "switching to temporal fallback across all stations."
    )
    temp_train_st, temp_test_st, temp_train_keys, temp_test_keys = make_split('temporal', selection=UNSEEN_STATION_SELECTION)
    ACTIVE_SPLIT_MODE = 'temporal'

    initialize_model_optimizer()
    base_temporal_result = run_training_for_split(
        split_label='temporal-base',
        train_station_set=temp_train_st,
        test_station_set=temp_test_st,
        train_key_set=temp_train_keys,
        test_key_set=temp_test_keys,
        hp=None,
        verbose=True,
    )
    best_temporal_result = base_temporal_result

    if RUN_OPTUNA:
        print("\n=== Optuna Tuning: Temporal Fallback ===")
        temporal_study = optuna.create_study(direction='maximize', study_name='temporal_r2_tuning_capped')
        temporal_study.optimize(objective_contingency_temporal, n_trials=N_TRIALS_CONTINGENCY, timeout=OPTUNA_TIMEOUT_SEC, show_progress_bar=False)
        best_temporal_trial = temporal_study.best_trial
        print(f"Best temporal trial (capped objective): {best_temporal_trial.value:.4f}")
        print(f"Best temporal params: {best_temporal_trial.params}")

        prev_frac = TEMPORAL_TEST_FRAC
        TEMPORAL_TEST_FRAC = best_temporal_trial.params.get('temp_test_frac', TEMPORAL_TEST_FRAC)
        tuned_temp_hp = suggest_hparams(best_temporal_trial)
        t_tr_st, t_te_st, t_tr_k, t_te_k = make_split('temporal', selection=UNSEEN_STATION_SELECTION)
        initialize_model_optimizer(tuned_temp_hp)
        tuned_temporal_result = run_training_for_split(
            split_label='temporal-tuned',
            train_station_set=t_tr_st,
            test_station_set=t_te_st,
            train_key_set=t_tr_k,
            test_key_set=t_te_k,
            hp=tuned_temp_hp,
            verbose=True,
        )
        TEMPORAL_TEST_FRAC = prev_frac

        if tuned_temporal_result['best_r2'] > best_temporal_result['best_r2']:
            best_temporal_result = tuned_temporal_result
            train_stations, test_stations = t_tr_st, t_te_st
            train_keys, test_keys = t_tr_k, t_te_k

    final_result = best_temporal_result
else:
    ACTIVE_SPLIT_MODE = 'station'
    final_result = best_unseen_result
if ALWAYS_REPORT_BOTH_SPLITS and best_temporal_result is None:
    print("\nCollecting contingency temporal metrics for reporting...")
    temp_train_st, temp_test_st, temp_train_keys, temp_test_keys = make_split('temporal', selection=UNSEEN_STATION_SELECTION)
    initialize_model_optimizer()
    best_temporal_result = run_training_for_split(
        split_label='temporal-report',
        train_station_set=temp_train_st,
        test_station_set=temp_test_st,
        train_key_set=temp_train_keys,
        test_key_set=temp_test_keys,
        hp=None,
        verbose=True,
    )

best_r2 = final_result['best_r2']
best_calib_coef = final_result['best_calib_coef']
best_anchor_r2 = final_result['best_anchor_r2']
best_epoch = final_result['best_epoch']
best_nrmse_pct = final_result.get('best_nrmse_pct', np.nan)
RESULT_BY_SPLIT = {
    'unseen': best_unseen_result,
    'temporal': best_temporal_result,
}

print(f"\nFinal chosen split: {ACTIVE_SPLIT_MODE}")
print(
    f"Best Val R2: {best_r2:.4f} | Best epoch: {best_epoch} | "
    f"Anchor R2: {best_anchor_r2:.4f} | NRMSE%: {best_nrmse_pct:.2f}"
)

print("\n=== Final Training Summary (Both Splits) ===")
if RESULT_BY_SPLIT.get('unseen') is not None:
    r = RESULT_BY_SPLIT['unseen']
    print(f"Unseen  | R2: {r['best_r2']:.4f} | Anchor R2: {r['best_anchor_r2']:.4f} | NRMSE%: {r.get('best_nrmse_pct', np.nan):.2f} | Epoch: {r['best_epoch']}")
else:
    print("Unseen  | not available")
if RESULT_BY_SPLIT.get('temporal') is not None:
    r = RESULT_BY_SPLIT['temporal']
    print(f"Temporal| R2: {r['best_r2']:.4f} | Anchor R2: {r['best_anchor_r2']:.4f} | NRMSE%: {r.get('best_nrmse_pct', np.nan):.2f} | Epoch: {r['best_epoch']}")
else:
    print("Temporal| not available")


=== Training Split: station-stable ===
Train stations: 24 | Test stations: 6
Train residual bounds: [-0.1315, 0.2324] from 17167/39480 valid train samples
Train kept samples: 16823/39480 (42.6%)
Train drop breakdown | out_of_split=9870, missing_label=22313, invalid_anchor=0, trimmed=344
Eval kept samples: 5676/9870 (57.5%)
Eval drop breakdown  | out_of_split=39480, missing_label=4194, invalid_anchor=0, trimmed=0
Prepared 17 train chunks
Prepared 3 eval chunks
Eval anchor mean/std: 0.2130/0.0377
Eval pred mean/std:   0.1875/0.0455
Eval true mean/std:   0.1875/0.0742
Anchor-only R2: 0.1791 | Calibrated R2: 0.3891 | NRMSE%: 5.80
Calib coef=[-0.1145368441939354, 2.8575546741485596, -2.842272996902466, -0.2560391128063202, -4.70504903793335, 0.005765202455222607]
Epoch 0 | Train Loss: 0.002863 | Val R2: 0.3891 | Anchor R2: 0.1791 | NRMSE%: 5.80 | coef=[-0.1145, 2.8576, -2.8423, -0.256, -4.705, 0.0058]
Epoch 1 | Train Loss: 0.002437 | Val R2: 0.4033 | Anchor R2: 0.1791 | NRMSE%: 5.73 | coef

[I 2026-04-19 14:19:55,777] A new study created in memory with name: unseen_r2_tuning


Epoch 15 | Train Loss: 0.001185 | Val R2: 0.3715 | Anchor R2: 0.1791 | NRMSE%: 5.88 | coef=[-0.1256, 1.7665, -0.6415, 6.0391, -1.2171, 3.3968]
Early stopping triggered at epoch 15 (patience=12).
Peak Val R2: 0.4133 at epoch 3 (anchor_R2=0.1791, NRMSE%=5.68, coef=[-0.2066, 2.8037, -0.9994, -0.9269, -4.4265, -1.4227])

=== Optuna Tuning: Unseen Split ===
Train residual bounds: [-0.1164, 0.1796] from 18662/39480 valid train samples
Train kept samples: 17234/39480 (43.7%)
Train drop breakdown | out_of_split=9870, missing_label=20818, invalid_anchor=0, trimmed=1428
Eval kept samples: 4181/9870 (42.4%)
Eval drop breakdown  | out_of_split=39480, missing_label=5689, invalid_anchor=0, trimmed=0
Epoch 0 | Train Loss: 0.002314 | Val R2: 0.5033 | Anchor R2: 0.4346 | NRMSE%: 4.86 | coef=[-0.0821, 1.2088, 0.0538, 10.0547, -0.7664, 4.0561]
Epoch 1 | Train Loss: 0.002092 | Val R2: 0.5304 | Anchor R2: 0.4346 | NRMSE%: 4.73 | coef=[0.0003, 0.7578, 1.4267, 6.6408, 0.8297, 3.1149]
Epoch 2 | Train Loss: 0.

[W 2026-04-19 14:20:04,623] Trial 0 failed with parameters: {'trim_low_q': 0.03995419008323581, 'trim_high_q': 0.9634574952147694, 'lr': 0.0005403217664261154, 'weight_decay': 1.3372656579623873e-06, 'huber_delta': 0.19250780513411986, 'target_chunk_train': 1024, 'target_chunk_eval': 1536, 'epochs': 26, 'patience': 9, 'min_r2_delta': 5.008941780513703e-05, 'calib_ridge': 0.00019455786911951606, 'unseen_selection': 'random'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_95693/3856718457.py", line 546, in objective_unseen
    res = run_training_for_split(
        split_label=f'unseen-{unseen_selection}',
    ...<5 lines>...
        verbose=False,
    )
  File "/tmp/ipykernel_95693/3856718457.py", line 438, in run_training_for_split
    pred_residual = model(x_batch, seq_batc

Epoch 9 | Train Loss: 0.000844 | Val R2: 0.4778 | Anchor R2: 0.4346 | NRMSE%: 4.98 | coef=[-0.0948, 1.7443, 1.5696, -7.2069, -1.5078, 7.611]


KeyboardInterrupt: 

: 

In [ ]:
def diagnose_sample_retention(
    cached_list,
    station_set=None,
    key_set=None,
    trim_low=None,
    trim_high=None,
    use_outlier_trim=True,
    label='split',
):
    cache = ensure_master_cache(cached_list)
    rows = cache['rows']
    meta = cache['meta']

    stats = {
        'total_rows': len(rows),
        'out_of_split': 0,
        'in_split': 0,
        'missing_label': 0,
        'invalid_anchor': 0,
        'trimmed': 0,
        'kept': 0,
    }

    for i, _ in enumerate(rows):
        pid, floored_ts, y_true, anchor_raw, valid_anchor = meta[i]

        if not keep_sample(pid, floored_ts, station_set, key_set):
            stats['out_of_split'] += 1
            continue

        stats['in_split'] += 1

        if y_true is None or np.isnan(y_true):
            stats['missing_label'] += 1
            continue

        if not valid_anchor:
            stats['invalid_anchor'] += 1
            continue

        residual_target = float(y_true - anchor_raw)
        if use_outlier_trim and trim_low is not None and trim_high is not None:
            if residual_target < trim_low or residual_target > trim_high:
                stats['trimmed'] += 1
                continue

        stats['kept'] += 1

    denom = max(stats['in_split'], 1)
    kept_pct = 100.0 * stats['kept'] / denom

    print(f"\n=== Retention Diagnostic: {label} ===")
    print(f"Total rows in cache:      {stats['total_rows']}")
    print(f"Out of split:             {stats['out_of_split']}")
    print(f"In split (denominator):   {stats['in_split']}")
    print(f"Missing label:            {stats['missing_label']}")
    print(f"Invalid anchor:           {stats['invalid_anchor']}")
    print(f"Trimmed by residual:      {stats['trimmed']}")
    print(f"Kept:                     {stats['kept']} ({kept_pct:.1f}%)")

    return stats

# Uses the same quantile bounds logic as training.
diag_trim_low, diag_trim_high = collect_train_residual_bounds(
    cached_data,
    station_set=train_stations,
    key_set=train_keys,
    low_q=TRIM_LOW_Q,
    high_q=TRIM_HIGH_Q,
)

train_diag = diagnose_sample_retention(
    cached_data,
    station_set=train_stations,
    key_set=train_keys,
    trim_low=diag_trim_low,
    trim_high=diag_trim_high,
    use_outlier_trim=ENABLE_OUTLIER_TRIM,
    label='TRAIN',
)

eval_diag = diagnose_sample_retention(
    cached_data,
    station_set=test_stations,
    key_set=test_keys,
    trim_low=diag_trim_low,
    trim_high=diag_trim_high,
    use_outlier_trim=(TRIM_EVAL_WITH_TRAIN_BOUNDS and ENABLE_OUTLIER_TRIM),
    label='EVAL',
)

Train residual bounds: [-0.1539, 0.2335] from 15641/15641 valid train samples

=== Retention Diagnostic: TRAIN ===
Total rows in cache:      49350
Out of split:             33709
In split (denominator):   15641
Missing label:            0
Invalid anchor:           0
Trimmed by residual:      314
Kept:                     15327 (98.0%)

=== Retention Diagnostic: EVAL ===
Total rows in cache:      49350
Out of split:             42148
In split (denominator):   7202
Missing label:            0
Invalid anchor:           0
Trimmed by residual:      0
Kept:                     7202 (100.0%)


In [ ]:
def evaluate_saved_split(label, split_result):
    if split_result is None:
        print(f"{label:<9} | not available")
        return None

    if split_result.get('best_state') is None:
        print(f"{label:<9} | missing model state")
        return None

    split_spec = split_result.get('split_eval_spec', {})
    test_station_set = split_spec.get('test_stations')
    test_key_set = split_spec.get('test_keys')

    model.load_state_dict(split_result['best_state'])
    model.eval()

    test_batches = build_eval_batches_for_epoch(
        cached_data,
        station_set=test_station_set,
        key_set=test_key_set,
        target_chunk_size=2048,
    )

    amp_enabled = (device.type == 'cuda') and (not REPRO_MODE)
    calib_coef = split_result['best_calib_coef']

    preds, actuals, anchors, residuals = [], [], [], []
    with torch.no_grad():
        for x_cpu, seq_cpu, y_cpu, anchor_cpu in test_batches:
            x_input = x_cpu.to(device, non_blocking=True)
            seq_input = seq_cpu.to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
                residual = model(x_input, seq_input).squeeze(1)

            residual_cpu = residual.float().cpu()
            pred = apply_quadratic_calibrator(anchor_cpu.numpy(), residual_cpu.numpy(), calib_coef)

            preds.extend(pred.tolist())
            actuals.extend(y_cpu.tolist())
            anchors.extend(anchor_cpu.tolist())
            residuals.extend(residual_cpu.tolist())

    if len(actuals) == 0:
        print(f"{label:<9} | no matched samples")
        return None

    r2 = r2_score(actuals, preds)
    anchor_r2 = r2_score(actuals, anchors)
    nrmse, nrmse_pct = compute_nrmse_percent(actuals, preds, max_possible_error=1.0)

    print(
        f"{label:<9} | R2: {r2:.4f} | Anchor R2: {anchor_r2:.4f} | "
        f"NRMSE: {nrmse:.6f} ({nrmse_pct:.2f}%)"
    )
    print(f"{label:<9} | Calibration coef: {[round(float(x), 4) for x in calib_coef.tolist()]}")
    print(f"{label:<9} | Actual mean/std: {np.mean(actuals):.4f}/{np.std(actuals):.4f}")
    print(f"{label:<9} | Pred mean/std:   {np.mean(preds):.4f}/{np.std(preds):.4f}")
    print(f"{label:<9} | Resid mean/std:  {np.mean(residuals):.4f}/{np.std(residuals):.4f}")

    return {
        'r2': r2,
        'anchor_r2': anchor_r2,
        'nrmse': nrmse,
        'nrmse_pct': nrmse_pct,
    }

print(f"Evaluation selected split: {ACTIVE_SPLIT_MODE}")
print("\n=== Final Eval Summary (Both Splits) ===")

unseen_eval = evaluate_saved_split('Unseen', RESULT_BY_SPLIT.get('unseen') if 'RESULT_BY_SPLIT' in globals() else None)
temporal_eval = evaluate_saved_split('Temporal', RESULT_BY_SPLIT.get('temporal') if 'RESULT_BY_SPLIT' in globals() else None)

FINAL_EVAL_BY_SPLIT = {
    'unseen': unseen_eval,
    'temporal': temporal_eval,
}


Evaluation selected split: temporal

=== Final Eval Summary (Both Splits) ===
Eval kept samples: 5676/9870 (57.5%)
Eval drop breakdown  | out_of_split=39480, missing_label=4194, invalid_anchor=0, trimmed=0
Unseen    | R2: 0.6211 | Anchor R2: 0.1791 | NRMSE: 0.045658 (4.57%)
Unseen    | Calibration coef: [-0.0782, 1.6759, 0.1058, 4.1928, -0.8497, -2.668]
Unseen    | Actual mean/std: 0.1875/0.0742
Unseen    | Pred mean/std:   0.1875/0.0580
Unseen    | Resid mean/std:  -0.0433/0.0334
Eval kept samples: 7202/7202 (100.0%)
Eval drop breakdown  | out_of_split=42148, missing_label=0, invalid_anchor=0, trimmed=0
Temporal  | R2: 0.7112 | Anchor R2: 0.3915 | NRMSE: 0.043112 (4.31%)
Temporal  | Calibration coef: [-0.0474, 1.4012, -0.4048, 6.3227, -0.8069, -0.4196]
Temporal  | Actual mean/std: 0.1978/0.0802
Temporal  | Pred mean/std:   0.1978/0.0676
Temporal  | Resid mean/std:  -0.0060/0.0547


In [ ]:
print("\n=== CHANNEL STATS ===")
sample_x, _, _ = cached_data[0]

for i in range(sample_x.shape[1]):
    mean = sample_x[:, i].mean().item()
    std = sample_x[:, i].std().item()
    print(f"Channel {i}: mean={mean:.3f}, std={std:.3f}")


=== CHANNEL STATS ===
Channel 0: mean=1.601, std=0.053
Channel 1: mean=1.596, std=0.154
Channel 2: mean=2.397, std=0.396
Channel 3: mean=1.135, std=0.166
Channel 4: mean=0.083, std=0.139
Channel 5: mean=0.939, std=0.760
Channel 6: mean=-0.610, std=0.160
Channel 7: mean=-1.299, std=0.014
Channel 8: mean=2.571, std=0.400
Channel 9: mean=-1.334, std=0.000
Channel 10: mean=-0.160, std=0.178
Channel 11: mean=0.199, std=1.168
Channel 12: mean=-0.214, std=1.257
Channel 13: mean=0.164, std=1.216
Channel 14: mean=0.214, std=1.123
Channel 15: mean=0.021, std=1.015
Channel 16: mean=0.011, std=0.994
Channel 17: mean=0.114, std=0.006
Channel 18: mean=-1.412, std=0.000


In [ ]:
dead_channels = (stds < 1e-5).nonzero().flatten().tolist()
print("Dead channels:", dead_channels)

Dead channels: []


In [ ]:
def check_tensor(x):
    if torch.isnan(x).any():
        print("NaNs detected")
    if torch.isinf(x).any():
        print("Infs detected")

for x, _, _ in cached_data[:5]:
    check_tensor(x)

In [ ]:
vals = list(gt_lookup.values())
print("\nTarget stats:")
print(f"Mean: {np.mean(vals):.4f}, Std: {np.std(vals):.4f}")
print(f"Min: {np.min(vals):.4f}, Max: {np.max(vals):.4f}")


Target stats:
Mean: 0.2159, Std: 0.0808
Min: 0.0000, Max: 0.4940


In [ ]:
print(f"Train stations: {len(train_stations)}")
print(f"Test stations: {len(test_stations)}")
print(f"Total samples: {sum(len(p) for _, p, _ in cached_data)}")

Train stations: 30
Test stations: 30
Total samples: 49350


In [ ]:
print("\n=== FEATURE ↔ TARGET CORR ===")
for i, f in enumerate(features):
    vals = []
    targets = []
    for x, pids, tss in cached_data:
        for j in range(len(pids)):
            key = (pids[j], int((tss[j].item() // 86400000) * 86400000))
            if key in gt_lookup:
                vals.append(x[j, i, PATCH_CENTER, PATCH_CENTER].item())
                targets.append(gt_lookup[key])
    if len(vals) > 0:
        corr = np.corrcoef(vals, targets)[0,1]
        print(f"{f}: {corr:.3f}")


=== FEATURE ↔ TARGET CORR ===
NDVI: -0.312
VV: -0.054
VH: -0.065
VV_VH_Ratio: -0.035
angle: -0.142
LST: -0.414
SMAP: 0.455
precip: 0.151
VPD: -0.447
API: 0.127
Albedo: 0.115
clay: 0.099
sand: -0.063
silt: 0.014
elevation: 0.069
slope: 0.150
aspect: 0.013
